# Kalshi Tennis Markets Exploration
Understanding how ATP match winner markets are priced on Kalshi.

In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt

BASE = 'https://api.elections.kalshi.com/trade-api/v2'

## 1. List active ATP match markets

In [ ]:
resp = requests.get(f'{BASE}/events', params={
    'limit': 100,
    'status': 'open',
    'series_ticker': 'KXATPMATCH',
})
events = resp.json().get('events', [])
print(f'Active ATP match events: {len(events)}')
for e in events:
    print(f"  {e['event_ticker']:50s} | {e['title']}")

## 2. Fetch market (YES/NO prices) for each match

In [ ]:
rows = []
for event in events:
    ticker = event['event_ticker']
    mkts = requests.get(f'{BASE}/markets', params={'event_ticker': ticker}).json().get('markets', [])
    for m in mkts:
        yes_bid = float(m.get('yes_bid_dollars', 0) or 0)
        yes_ask = float(m.get('yes_ask_dollars', 0) or 0)
        rows.append({
            'event':      event['title'],
            'ticker':     m['ticker'],
            'outcome':    m.get('yes_sub_title', m.get('title', '')),
            'yes_bid':    yes_bid,
            'yes_ask':    yes_ask,
            'mid':        (yes_bid + yes_ask) / 2 if yes_ask > 0 else None,
            'spread':     round(yes_ask - yes_bid, 4) if yes_ask > 0 else None,
            'volume':     float(m.get('volume_fp', 0) or 0),
            'close_time': m.get('close_time', ''),
        })

markets_df = pd.DataFrame(rows)
markets_df

## 3. Implied probability distribution
Each YES price = market's implied probability that player wins.

In [ ]:
# Filter to markets where both sides are priced
priced = markets_df[markets_df['mid'].notna() & (markets_df['mid'] > 0)].copy()

priced['mid'].hist(bins=20, edgecolor='black')
plt.xlabel('Implied win probability (YES mid price)')
plt.title('Distribution of Kalshi ATP match prices')
plt.tight_layout()

## 4. Spread analysis — how much edge does the house take?

In [ ]:
print(f'Avg spread : {priced["spread"].mean():.4f} ({priced["spread"].mean()*100:.2f}¢ per $1)')
print(f'Min spread : {priced["spread"].min():.4f}')
print(f'Max spread : {priced["spread"].max():.4f}')
print(f'\nNote: to profit, your model needs edge > spread/2')

## 5. Inspect a single match in detail

In [ ]:
# Pick the first event
sample_event = events[0]
print('Match:', sample_event['title'])

ticker = sample_event['event_ticker']
mkts   = requests.get(f'{BASE}/markets', params={'event_ticker': ticker}).json().get('markets', [])

for m in mkts:
    print(f"\n  Outcome : {m.get('yes_sub_title', m.get('title', ''))}")
    print(f"  YES bid : ${float(m.get('yes_bid_dollars', 0) or 0):.4f}")
    print(f"  YES ask : ${float(m.get('yes_ask_dollars', 0) or 0):.4f}")
    print(f"  Volume  : {float(m.get('volume_fp', 0) or 0):.2f}")
    print(f"  Closes  : {m.get('close_time', 'N/A')}")

## 6. French Open 2026 winner market

In [ ]:
fo_event = 'KXFOMEN-26'
fo_markets = requests.get(f'{BASE}/markets', params={'event_ticker': fo_event, 'limit': 100}).json().get('markets', [])

fo_df = pd.DataFrame([{
    'player':   m.get('yes_sub_title', m.get('title', '')),
    'yes_bid':  float(m.get('yes_bid_dollars', 0) or 0),
    'yes_ask':  float(m.get('yes_ask_dollars', 0) or 0),
    'mid':      (float(m.get('yes_bid_dollars', 0) or 0) + float(m.get('yes_ask_dollars', 0) or 0)) / 2,
    'volume':   float(m.get('volume_fp', 0) or 0),
} for m in fo_markets])

fo_df = fo_df[fo_df['mid'] > 0].sort_values('mid', ascending=False)
print(f'French Open field markets: {len(fo_df)}')

fo_df.set_index('player')['mid'].plot(
    kind='barh',
    title='2026 French Open implied win probability (Kalshi)',
    xlabel='Implied probability',
    figsize=(10, 8)
)
plt.tight_layout()

## 7. ATP Challenger markets

In [ ]:
ch_events = requests.get(f'{BASE}/events', params={
    'limit': 50, 'status': 'open', 'series_ticker': 'KXATPCHALLENGERMATCH'
}).json().get('events', [])

print(f'Active Challenger events: {len(ch_events)}')
for e in ch_events:
    print(f"  {e['event_ticker']:50s} | {e['title']}")